# Lesson 14b: Hierarchical RL Practical

**Implementations:**
- Options framework
- Feudal manager-worker architecture
- Goal-conditioned policies
- Hindsight Experience Replay (HER)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from collections import deque
import random

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Options Implementation

In [ ]:
class Option:
    """Single option (policy, initiation, termination)."""
    
    def __init__(self, policy, initiation_set=None, termination_prob=0.1):
        self.policy = policy
        self.initiation_set = initiation_set
        self.termination_prob = termination_prob
    
    def can_initiate(self, state):
        if self.initiation_set is None:
            return True
        return state in self.initiation_set
    
    def should_terminate(self, state):
        return np.random.rand() < self.termination_prob
    
    def get_action(self, state):
        return self.policy(state)

class OptionCritic:
    """Option-Critic algorithm."""
    
    def __init__(self, state_dim, n_actions, n_options, lr=1e-3):
        self.n_options = n_options
        
        # Option policies
        self.option_policies = nn.ModuleList([
            nn.Sequential(
                nn.Linear(state_dim, 64),
                nn.ReLU(),
                nn.Linear(64, n_actions)
            ) for _ in range(n_options)
        ]).to(device)
        
        # Q over options
        self.q_omega = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, n_options)
        ).to(device)
        
        self.optimizer = torch.optim.Adam(
            list(self.option_policies.parameters()) + list(self.q_omega.parameters()),
            lr=lr
        )
    
    def select_option(self, state, epsilon=0.1):
        if np.random.rand() < epsilon:
            return np.random.randint(self.n_options)
        with torch.no_grad():
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
            q_values = self.q_omega(state_tensor)
            return q_values.argmax().item()
    
    def select_action(self, state, option):
        with torch.no_grad():
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
            action_probs = F.softmax(self.option_policies[option](state_tensor), dim=-1)
            return torch.multinomial(action_probs, 1).item()

print("Option-Critic implemented")

## 2. Feudal Networks

In [ ]:
class FeudalNetwork:
    """Manager-Worker hierarchy."""
    
    def __init__(self, state_dim, action_dim, goal_dim=16, c=10):
        self.c = c  # Manager time scale
        
        # Manager (sets goals)
        self.manager = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, goal_dim)
        ).to(device)
        
        # Worker (follows goals)
        self.worker = nn.Sequential(
            nn.Linear(state_dim + goal_dim, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim)
        ).to(device)
        
        self.optimizer_manager = torch.optim.Adam(self.manager.parameters(), lr=1e-3)
        self.optimizer_worker = torch.optim.Adam(self.worker.parameters(), lr=1e-3)
    
    def select_action(self, state, goal=None):
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(device)
        
        if goal is None:
            goal = self.manager(state_tensor)
        
        worker_input = torch.cat([state_tensor, goal], dim=-1)
        action_logits = self.worker(worker_input)
        action_probs = F.softmax(action_logits, dim=-1)
        
        return torch.multinomial(action_probs, 1).item(), goal

print("Feudal Network implemented")

## 3. Goal-Conditioned Policy with HER

In [ ]:
class GoalConditionedPolicy:
    """Policy conditioned on goals."""
    
    def __init__(self, state_dim, action_dim, goal_dim):
        self.policy = nn.Sequential(
            nn.Linear(state_dim + goal_dim, 128),
            nn.ReLU(),
            nn.Linear(128, action_dim)
        ).to(device)
        
        self.optimizer = torch.optim.Adam(self.policy.parameters(), lr=1e-3)
    
    def select_action(self, state, goal):
        state_goal = torch.FloatTensor(np.concatenate([state, goal])).unsqueeze(0).to(device)
        action_probs = F.softmax(self.policy(state_goal), dim=-1)
        return torch.multinomial(action_probs, 1).item()

class HindsightBuffer:
    """Replay buffer with hindsight relabeling."""
    
    def __init__(self, capacity=10000):
        self.buffer = deque(maxlen=capacity)
    
    def push_episode(self, episode):
        """Store episode and create hindsight samples."""
        # Store original
        self.buffer.extend(episode)
        
        # Hindsight: relabel with achieved goals
        for i, (s, a, r, s_next, g, done) in enumerate(episode):
            # Future strategy: sample achieved goal from later in episode
            if i < len(episode) - 1:
                future_idx = np.random.randint(i + 1, len(episode))
                achieved_goal = episode[future_idx][0]  # Use future state as goal
                
                # Compute reward w.r.t. achieved goal
                new_reward = 0 if np.linalg.norm(s_next - achieved_goal) > 0.1 else 1
                self.buffer.append((s, a, new_reward, s_next, achieved_goal, done))
    
    def sample(self, batch_size):
        return random.sample(self.buffer, min(batch_size, len(self.buffer)))

print("Goal-conditioned policy + HER implemented")

## Summary

**Implemented:**
1. Options framework with option-critic
2. Feudal manager-worker hierarchy
3. Goal-conditioned policies
4. Hindsight Experience Replay

**Benefits:**
- Temporal abstraction
- Hierarchical planning
- Skill reuse
- Sample efficiency (HER)

**Next:** Lesson 15 - Meta-Learning